In [ ]:
# See nb_gold_build for why pyodbc + AAD token from inside a Fabric
# Spark notebook is the chosen way to run T-SQL here. Every cell writes
# its result (or, on failure, the full traceback) to Files/diagnostics/
# so failures are self-diagnosable via the OneLake DFS API -- the Fabric
# REST job-status API never surfaces cell-level errors.
import pyodbc, struct, json as _json, traceback as _tb

WORKSPACE_ID = "16b2a9c6-6418-485f-865f-c47442b8b6c6"
LH_BRONZE_ID = "09a8b1f5-f33c-4eed-912d-40b9cbdc9b06"

def write_diag(name, obj):
    path = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LH_BRONZE_ID}/Files/diagnostics/{name}.json"
    notebookutils.fs.put(path, _json.dumps(obj, indent=2, default=str), overwrite=True)
    print(f"wrote {path}")

try:
    server = "4oa7g2g2i24uposxn4zcxdynue-y2u3efqymrpurbs7yr2efofwyy.datawarehouse.fabric.microsoft.com"
    database = "wh_gold"
    token = notebookutils.credentials.getToken("https://database.windows.net/")
    token_bytes = token.encode("utf-16-le")
    token_struct = struct.pack(f"<I{len(token_bytes)}s", len(token_bytes), token_bytes)
    SQL_COPT_SS_ACCESS_TOKEN = 1256
    conn_str = f"DRIVER={{ODBC Driver 18 for SQL Server}};SERVER={server};DATABASE={database};Encrypt=yes;"
    conn = pyodbc.connect(conn_str, attrs_before={SQL_COPT_SS_ACCESS_TOKEN: token_struct}, autocommit=True)
    cursor = conn.cursor()
    print("connected")
    write_diag("gold_turns_connect", {"status": "ok"})
except Exception as e:
    write_diag("gold_turns_connect", {"status": "failed", "error": repr(e), "traceback": _tb.format_exc()})
    raise


In [ ]:
try:
    cursor.execute("DROP TABLE IF EXISTS dbo.fact_aircraft_turn")
    cursor.execute("""
        CREATE TABLE dbo.fact_aircraft_turn (
            turn_key BIGINT NOT NULL,
            tail_number VARCHAR(10) NOT NULL,
            station VARCHAR(3) NOT NULL,
            inbound_arr_sched_utc DATETIME2(6) NULL,
            inbound_arr_actual_utc DATETIME2(6) NULL,
            outbound_dep_sched_utc DATETIME2(6) NULL,
            outbound_dep_actual_utc DATETIME2(6) NULL,
            scheduled_turn_min FLOAT NULL,
            actual_turn_min FLOAT NOT NULL,
            inbound_arr_delay_min FLOAT NOT NULL,
            outbound_dep_delay_min FLOAT NOT NULL,
            minutes_recovered FLOAT NOT NULL,
            fleet_family VARCHAR(50) NULL,
            turn_below_fleet_p10 BIT NULL
        )
    """)
    print("DDL done")
except Exception as e:
    write_diag('gold_turns_ddl_ERROR', {'error': repr(e), 'traceback': _tb.format_exc()})
    raise


In [ ]:
try:
    cursor.execute("""
        WITH ordered_legs AS (
            SELECT
                tail_number,
                origin_airport_key,
                dest_airport_key,
                sched_dep_utc,
                sched_arr_utc,
                actual_dep_utc,
                actual_arr_utc,
                dep_delay_min,
                arr_delay_min,
                COALESCE(actual_dep_utc, sched_dep_utc) AS seq_key
            FROM dbo.fact_flight_leg
            WHERE tail_number IS NOT NULL
        ),
        paired AS (
            SELECT
                tail_number,
                dest_airport_key AS inbound_dest,
                sched_arr_utc AS inbound_arr_sched,
                actual_arr_utc AS inbound_arr_actual,
                COALESCE(arr_delay_min, 0) AS inbound_arr_delay,
                LEAD(origin_airport_key) OVER (PARTITION BY tail_number ORDER BY seq_key) AS outbound_origin,
                LEAD(sched_dep_utc) OVER (PARTITION BY tail_number ORDER BY seq_key) AS outbound_dep_sched,
                LEAD(actual_dep_utc) OVER (PARTITION BY tail_number ORDER BY seq_key) AS outbound_dep_actual,
                LEAD(COALESCE(dep_delay_min, 0)) OVER (PARTITION BY tail_number ORDER BY seq_key) AS outbound_dep_delay
            FROM ordered_legs
        )
        INSERT INTO dbo.fact_aircraft_turn (
            turn_key, tail_number, station, inbound_arr_sched_utc, inbound_arr_actual_utc,
            outbound_dep_sched_utc, outbound_dep_actual_utc, scheduled_turn_min, actual_turn_min,
            inbound_arr_delay_min, outbound_dep_delay_min, minutes_recovered
        )
        SELECT
            ROW_NUMBER() OVER (ORDER BY tail_number, inbound_arr_actual),
            tail_number,
            inbound_dest,
            inbound_arr_sched,
            inbound_arr_actual,
            outbound_dep_sched,
            outbound_dep_actual,
            CASE WHEN inbound_arr_sched IS NOT NULL AND outbound_dep_sched IS NOT NULL
                 THEN DATEDIFF(SECOND, inbound_arr_sched, outbound_dep_sched) / 60.0 END,
            DATEDIFF(SECOND, inbound_arr_actual, outbound_dep_actual) / 60.0,
            inbound_arr_delay,
            outbound_dep_delay,
            inbound_arr_delay - outbound_dep_delay
        FROM paired
        WHERE inbound_dest = outbound_origin
          AND inbound_arr_actual IS NOT NULL
          AND outbound_dep_actual IS NOT NULL
          AND DATEDIFF(SECOND, inbound_arr_actual, outbound_dep_actual) >= 0
          AND DATEDIFF(SECOND, inbound_arr_actual, outbound_dep_actual) <= 21600
    """)
    cursor.execute("SELECT COUNT(*) FROM dbo.fact_aircraft_turn")
    turn_count = cursor.fetchone()[0]
    print(f"fact_aircraft_turn: {turn_count:,} rows")
    write_diag("gold_turns_load", {"rows": turn_count})
except Exception as e:
    write_diag('gold_turns_load_wrapped_ERROR', {'error': repr(e), 'traceback': _tb.format_exc()})
    raise


In [ ]:
try:
    cursor.execute("""
        ;WITH flagged AS (
            SELECT
                t.turn_key,
                a.fleet_family,
                PERCENTILE_CONT(0.1) WITHIN GROUP (ORDER BY t.actual_turn_min)
                    OVER (PARTITION BY a.fleet_family) AS p10_min,
                t.actual_turn_min
            FROM dbo.fact_aircraft_turn t
            JOIN dbo.dim_aircraft a ON t.tail_number = a.tail_number
        )
        UPDATE t
        SET t.fleet_family = f.fleet_family,
            t.turn_below_fleet_p10 = CASE WHEN f.actual_turn_min < f.p10_min THEN 1 ELSE 0 END
        FROM dbo.fact_aircraft_turn t
        JOIN flagged f ON t.turn_key = f.turn_key
    """)
    cursor.execute("SELECT COUNT(*) FROM dbo.fact_aircraft_turn WHERE fleet_family IS NOT NULL")
    flagged_count = cursor.fetchone()[0]
    print(f"fleet_family/turn_below_fleet_p10 set on {flagged_count:,} rows")
    write_diag("gold_turns_fleet_p10", {"status": "ok", "flagged_rows": flagged_count})
except Exception as e:
    print("fleet p10 step FAILED (non-fatal, core pairing already committed):", repr(e))
    write_diag("gold_turns_fleet_p10", {"status": "failed", "error": repr(e)})


In [ ]:
try:
    # Hand-check: plan's Phase 2 "done when" asked for 20 hand-checked DFW
    # turns pairing correctly -- sample 20 real DFW turns with enough detail
    # to eyeball-verify the pairing (same tail, same station, sane gap).
    cursor.execute("""
        SELECT TOP 20 tail_number, station, inbound_arr_sched_utc, inbound_arr_actual_utc,
               outbound_dep_sched_utc, outbound_dep_actual_utc, actual_turn_min, minutes_recovered
        FROM dbo.fact_aircraft_turn
        WHERE station = 'DFW'
        ORDER BY inbound_arr_actual_utc DESC
    """)
    rows = cursor.fetchall()
    sample = [
        {
            "tail_number": r[0], "station": r[1],
            "inbound_arr_sched_utc": str(r[2]), "inbound_arr_actual_utc": str(r[3]),
            "outbound_dep_sched_utc": str(r[4]), "outbound_dep_actual_utc": str(r[5]),
            "actual_turn_min": r[6], "minutes_recovered": r[7],
        }
        for r in rows
    ]
    cursor.execute("SELECT COUNT(*) FROM dbo.fact_aircraft_turn WHERE station = 'DFW'")
    dfw_turn_count = cursor.fetchone()[0]
    print(f"DFW turns: {dfw_turn_count:,}")
    write_diag("gold_turns_dfw_sample", {"dfw_turn_count": dfw_turn_count, "sample_20": sample})
except Exception as e:
    write_diag('gold_turns_dfw_sample_wrapped_ERROR', {'error': repr(e), 'traceback': _tb.format_exc()})
    raise
